In [ ]:
import numpy as np
import pandas as pd
import scipy.special
import scipy.interpolate
from numpy.typing import NDArray

In [ ]:
def compute_weibull_U_from_A(A: float, k: float) -> float:
    """Compute the mean of the Weibull distribution from its parameters.

    Parameters
    ----------
    A
        Scale parameter of the Weibull distribution.
    k
        Shape parameter of the Weibull distribution.

    Returns
    -------
    float
        Mean of the Weibull distribution.
    """
    return A * scipy.special.gamma(1 + 1 / k)


def compute_weibull_A_from_U(U: float, k: float) -> float:
    """Compute the scale parameter from mean and shape parameter of the Weibull distribution.

    Parameters
    ----------
    U
        Mean of the Weibull distribution.
    k
        Shape parameter of the Weibull distribution.

    Returns
    -------
    float
        Scale parameter of the Weibull distribution.
    """
    return U / scipy.special.gamma(1 + 1 / k)


def generate_random_weibull(size: tuple[int, ...], A: float, k: float, seed: int | None = None) -> NDArray[np.floating]:
    rng = np.random.default_rng(seed=seed)
    X = rng.random(size=size)
    wind_speeds = np.float_power(-np.log(X), 1 / k) * A
    return wind_speeds


def wind_speed_to_power(
    nominal_power: float = 5600, cut_in: float = 3.0, cut_out: float = 25.0, rated_wind_speed: float = 13.0
) -> NDArray[np.floating]:
    t = np.array([0, cut_in - 0.5, cut_in, rated_wind_speed, cut_out, cut_out + 0.5, 30])
    p = np.array([0, 0, 0.04 * nominal_power, nominal_power, nominal_power, 0, 0])
    grid = np.linspace(0, 30, num=61)
    values = scipy.interpolate.make_interp_spline(t, p, k=1)(grid)
    return np.column_stack([grid, values])


def generate_wtg_scada(A=10, k=2):
    index = pd.date_range("2026-02-14T07:00:00", freq="10min", periods=5)
    wind_speed = generate_random_weibull(size=(len(index)), A=A, k=k)
    df = pd.DataFrame(data={"wind_speed": wind_speed}, index=index)

    powercurve = wind_speed_to_power()
    power = scipy.interpolate.make_interp_spline(powercurve[:, 0], powercurve[:, 1], k=1)(wind_speed)
    df["power"] = power
    return df

In [ ]:
A, k = 10, 2
print(compute_weibull_U_from_A(A=A, k=k))
Us = generate_random_weibull(size=(10,), A=A, k=k)
print(Us)
print(Us.mean())

In [ ]:
generate_wtg_scada()